# DC-Ops: Export YOLOv8n-seg to QNN HTP for Snapdragon NPU

Exports the trained YOLOv8n-seg detection head to ExecuTorch .pte with QNN HTP
backend (SM8750). Outputs a single detection tensor [1, 52, 8400] that the app's
existing parseYoloOutput() already handles (bbox path).

**Runtime → A100 GPU (High-RAM for faster build)** (Linux required for QNN compilation)

In [ ]:
# 1. Install ExecuTorch + QNN SDK
!pip install -q ultralytics py-cpuinfo huggingface_hub
!git clone --depth 1 https://github.com/pytorch/executorch.git
%cd executorch
!git submodule sync
!git submodule update --init --recursive --depth 1
!bash install_requirements.sh
!pip install -e . --no-build-isolation

!python backends/qualcomm/scripts/download_qnn_sdk.py --print-sdk-path 2>/dev/null || true
import os
qnn_path = !python backends/qualcomm/scripts/download_qnn_sdk.py --print-sdk-path 2>/dev/null
os.environ['QNN_SDK_ROOT'] = qnn_path[-1].strip() if qnn_path else ''
print(f'QNN SDK: {os.environ.get("QNN_SDK_ROOT", "NOT SET")}')

In [ ]:
# 2. Download trained YOLOv8n-seg from HuggingFace
import os
os.chdir('/content')
from huggingface_hub import hf_hub_download

model_path = hf_hub_download(
    repo_id='abhijitbetigeri/dc-ops-dataset',
    filename='models/dc_ops_yolov8n_seg_v3.pt',
    repo_type='dataset',
)
print(f'Model: {model_path}')

In [ ]:
# 3. Load YOLO and wrap to output ONLY the detection tensor [1, 52, 8400]
import torch
from ultralytics import YOLO

yolo = YOLO(model_path)
core = yolo.model.eval()

class YoloDetWrapper(torch.nn.Module):
    """Returns just the detection tensor (4 bbox + 16 cls + 32 mask coeffs = 52).
    App uses the bbox path; no mask protos needed for the demo."""
    def __init__(self, model):
        super().__init__()
        self.model = model
    def forward(self, x):
        out = self.model(x)
        # nested: out[0] = (det_tensor, mask_protos), out[1] = dict
        if isinstance(out, (list, tuple)):
            return out[0][0]
        return out

wrapper = YoloDetWrapper(core).eval()

# Sanity check output shape
with torch.no_grad():
    test = wrapper(torch.randn(1, 3, 640, 640))
print(f'Detection output shape: {tuple(test.shape)}')   # expect (1, 52, 8400)

In [ ]:
# 4. Load QNN libs and export with HTP backend (INT8)
import os, sys, ctypes, glob
os.chdir('/content/executorch')
sys.path.insert(0, '/content/executorch')

qnn_sdk = os.environ.get('QNN_SDK_ROOT', '')
os.environ['LD_LIBRARY_PATH'] = f"{qnn_sdk}/lib/x86_64-linux-clang/:{os.environ.get('LD_LIBRARY_PATH', '')}"
for lib in sorted(glob.glob(f"{qnn_sdk}/lib/x86_64-linux-clang/libQnn*.so")):
    try:
        ctypes.CDLL(lib)
    except: pass
print('QNN libs loaded')

import torch
from executorch.backends.qualcomm.export_utils import build_executorch_binary, QnnConfig
from executorch.backends.qualcomm.quantizer.quantizer import QuantDtype

# Calibration data
calib_inputs = [(torch.randn(1, 3, 640, 640),) for _ in range(20)]

qnn_config = QnnConfig(
    soc_model="SM8750",
    backend="htp",
    build_folder="/content/qnn_build",
    compile_only=True,
)

print('Building ExecuTorch binary with QNN HTP backend for YOLOv8n-seg...')
build_executorch_binary(
    model=wrapper,
    qnn_config=qnn_config,
    file_name='/content/dc_ops_yolo_qnn',
    dataset=calib_inputs,
    quant_dtype=QuantDtype.use_8a8w,
)
print('QNN export done!')

In [ ]:
# 5. Find and download the .pte
import os, glob
ptes = [p for p in glob.glob('/content/**/*.pte', recursive=True) if 'yolo' in p.lower() or 'dc_ops' in p.lower()]
print('Found:', ptes)
for p in ptes:
    print(f'  {p}: {os.path.getsize(p)/1e6:.1f} MB')

from google.colab import files
for p in ptes:
    files.download(p)